In [1]:
# =============================================================================
# STEP 6 - CLASS-CONDITIONAL COVARIATE SHIFT
#
# The study measures feature shift with ONE domain classifier over the whole
# dataset, then uses that single number to explain class-conditional coverage.
# The paper already contains the refutation: UGR'16 has one aggregate S_cov of
# 0.69, under which scan11 collapses to 0.535 while dos holds at 0.950. A single
# aggregate statistic cannot distinguish those classes, so it cannot predict them.
#
# This computes, for every class c in every environment,
#
#     S_cov,c = cross-fitted AUROC of a domain classifier trained to separate
#               source X | Y=c  from  target X | Y=c
#
# and then asks whether class-specific distinguishability, class-specific subtype
# novelty and score movement jointly order class-specific undercoverage.
#
# The expected payoff is a stronger central claim: aggregate shift statistics fail
# not because shift is hard to measure but because the relevant shift is
# class-conditional. That is more coherent than the present framing, which uses an
# aggregate measure in a decomposition of class-level outcomes.
#
# It can also fail. If S_cov,c does not order undercoverage either, then feature
# distinguishability is not the operative quantity at any resolution and only score
# movement is, which is a different and narrower paper.
# =============================================================================
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess, hashlib, time
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
import importlib
for m in ['config','conformal']:
    if m in sys.modules: importlib.reload(sys.modules[m])
import config, conformal
import numpy as np, pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
RD=config.REPORTS_DIR; ALPHA=config.ALPHA_PRIMARY
SUB=20000; FOLDS=5; MIN_PER_SIDE=40      # below this a class-conditional AUROC is not estimable
print('ready | alpha', ALPHA, '| per-side cap', SUB, '| folds', FOLDS)


Mounted at /content/drive
ready | alpha 0.05 | per-side cap 20000 | folds 5


In [2]:
# =============================================================================
# Cell 2 - the estimator, and a permutation null so "no class-conditional shift"
# has a reference rather than being read off an absolute number.
# =============================================================================
def scov(Xs, Xt, seed=0, folds=FOLDS):
    """Cross-fitted domain-classifier AUROC between two feature matrices."""
    ns_, nt_ = len(Xs), len(Xt)
    if ns_ < MIN_PER_SIDE or nt_ < MIN_PER_SIDE: return np.nan, ns_, nt_
    m = min(ns_, nt_, SUB)
    rg = np.random.default_rng(seed)
    a = Xs[rg.choice(ns_, m, replace=False)]
    b = Xt[rg.choice(nt_, m, replace=False)]
    X = np.vstack([a, b]); y = np.r_[np.zeros(m), np.ones(m)]
    k = min(folds, m // 10) if m < 10*folds else folds
    if k < 2: return np.nan, ns_, nt_
    aucs=[]
    for tr, te in StratifiedKFold(k, shuffle=True, random_state=seed).split(X, y):
        mod = HistGradientBoostingClassifier(max_iter=100, random_state=seed).fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], mod.predict_proba(X[te])[:, 1]))
    return float(np.mean(aucs)), ns_, nt_

def scov_null(Xs, Xt, seed=0, draws=5):
    """Permutation reference: split the POOLED data at random, so any AUROC above
    this reflects genuine source/target separation rather than estimator optimism."""
    if len(Xs) < MIN_PER_SIDE or len(Xt) < MIN_PER_SIDE: return np.nan
    P = np.vstack([Xs, Xt]); out=[]
    for d in range(draws):
        rg = np.random.default_rng(seed + 1000 + d)
        perm = rg.permutation(len(P)); half = len(P)//2
        v,_,_ = scov(P[perm[:half]], P[perm[half:]], seed=seed+d, folds=3)
        if not np.isnan(v): out.append(v)
    return float(np.mean(out)) if out else np.nan
print('estimator and permutation null defined')
print(f'  classes with fewer than {MIN_PER_SIDE} rows on either side are reported as')
print('  not estimable rather than given a number that cannot be trusted')


estimator and permutation null defined
  classes with fewer than 40 rows on either side are reported as
  not estimable rather than given a number that cannot be trusted


In [3]:
# =============================================================================
# Cell 3 - build the per-class source and target feature matrices for all four
# environments, using the same partitions the coverage analysis used.
# =============================================================================
ENVS={}

# ---------- NSL-KDD ----------
CL=config.CANONICAL_CLASSES
tr=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
te=pd.read_parquet(config.INTERIM_DIR/'nslkdd_test.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
tr=tr.assign(partition=part['partition'].values)
DROP={'label','subtype','partition','is_unseen'}
F=[c for c in tr.columns if c not in DROP and pd.api.types.is_numeric_dtype(tr[c])]
src=tr[tr.partition=='source_cal_pool']
ENVS['nslkdd']={'feats':F,
   'src':{c: src[src.label==c][F].to_numpy(float) for c in CL},
   'tgt':{c: te[te.label==c][F].to_numpy(float) for c in CL}}

# ---------- UGR'16 ----------
UGR=config.DATASETS_DIR/'ugr16'
us=pd.read_parquet(UGR/'july_week5.parquet'); ut=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (us,ut): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
us=us[us.label.isin(UK)].reset_index(drop=True); ut=ut[ut.label.isin(UK)].reset_index(drop=True)
FU=[c for c in us.columns if c not in DROP and pd.api.types.is_numeric_dtype(us[c])]
def strat(df,fr,seed,col='label'):
    rg=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rg.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
us=us.assign(partition=strat(us,config.SPLIT_FRACTIONS,20260725).values)
usp=us[us.partition=='source_cal_pool']
ENVS['ugr16']={'feats':FU,
   'src':{c: usp[usp.label==c][FU].to_numpy(float) for c in sorted(UK)},
   'tgt':{c: ut[ut.label==c][FU].to_numpy(float) for c in sorted(UK)}}

# ---------- CIC-IoT-2023 ----------
iot=pd.read_parquet(config.PROC_DIR/'ciciot2023_prepared.parquet')
sp=pd.read_parquet(config.PROC_DIR/'ciciot2023_split.parquet')
iot['side']=sp['side'].values; iot['partition']=sp['partition'].values
FI=json.loads((RD/'ciciot2023_prepared_fingerprint.json').read_text())['features']
isp=iot[iot.partition=='source_cal_pool']; itg=iot[iot.partition=='target_pool']
ICL=json.loads((RD/'ciciot2023_model_record.json').read_text())['classes_canonical_order']
ENVS['ciciot2023']={'feats':FI,
   'src':{c: isp[isp.family==c][FI].to_numpy(float) for c in ICL},
   'tgt':{c: itg[itg.family==c][FI].to_numpy(float) for c in ICL}}

for k,v in ENVS.items():
    print(f"{k:12s} {len(v['feats']):3d} features | classes " +
          ", ".join(f"{c}({len(v['src'][c])}/{len(v['tgt'][c])})" for c in v['src']))
print("\n(counts are source/target rows per class; CIC-IDS2017 uses a within-day")
print(" holdout whose per-class source/target split is realization-specific and is")
print(" therefore handled separately in cell 4)")


nslkdd        38 features | classes Normal(10101/9711), DoS(6889/7460), Probe(1748/2421), R2L(149/2885), U2R(7/67)
ugr16          7 features | classes background(30000/200000), dos(7500/50000), nerisbotnet(7500/50000), scan11(7500/50000), scan44(7500/50000)
ciciot2023    44 features | classes Benign(6299/17999), BruteForce(1371/3919), DDoS(68476/195662), DoS(25215/72046), Mirai(18953/54156), Recon(23055/65879), Spoofing(12575/35934), Web(2135/10579)

(counts are source/target rows per class; CIC-IDS2017 uses a within-day
 holdout whose per-class source/target split is realization-specific and is
 therefore handled separately in cell 4)


In [4]:
# =============================================================================
# Cell 4 - compute S_cov,c for every class, plus the permutation null and the
# aggregate value for comparison.
# =============================================================================
rows=[]; t0=time.time()
for env,d in ENVS.items():
    # aggregate, all classes pooled, matching the study's existing measure
    Xs=np.vstack([v for v in d['src'].values() if len(v)]);
    Xt=np.vstack([v for v in d['tgt'].values() if len(v)])
    agg,_,_=scov(Xs,Xt,seed=7)
    print(f'{env}: aggregate S_cov = {agg:.4f}')
    for c in d['src']:
        v,ns_,nt_=scov(d['src'][c], d['tgt'][c], seed=hash((env,c))%(2**31))
        nul=scov_null(d['src'][c], d['tgt'][c], seed=hash((env,c))%(2**31))
        rows.append({'dataset':env,'class':c,'S_cov_class':v,'S_cov_null':nul,
                     'S_cov_aggregate':agg,'n_src':ns_,'n_tgt':nt_,
                     'estimable':bool(not np.isnan(v))})
        tag = f'{v:.4f}' if not np.isnan(v) else 'not estimable'
        nn  = f' (null {nul:.3f})' if not np.isnan(nul) else ''
        print(f'   {c:14s} S_cov,c = {tag}{nn}   n={ns_}/{nt_}')
SC=pd.DataFrame(rows)
print(f'\n{len(SC)} class cells | {time.time()-t0:.0f}s')
print(f'estimable: {int(SC.estimable.sum())} | not estimable: {int((~SC.estimable).sum())}')


nslkdd: aggregate S_cov = 0.8934
   Normal         S_cov,c = 0.7886 (null 0.501)   n=10101/9711
   DoS            S_cov,c = 0.9595 (null 0.499)   n=6889/7460
   Probe          S_cov,c = 0.9794 (null 0.497)   n=1748/2421
   R2L            S_cov,c = 0.9905 (null 0.492)   n=149/2885
   U2R            S_cov,c = not estimable   n=7/67
ugr16: aggregate S_cov = 0.7129
   background     S_cov,c = 0.6696 (null 0.498)   n=30000/200000
   dos            S_cov,c = 0.5424 (null 0.500)   n=7500/50000
   nerisbotnet    S_cov,c = 0.5782 (null 0.499)   n=7500/50000
   scan11         S_cov,c = 0.9997 (null 0.498)   n=7500/50000
   scan44         S_cov,c = 0.9861 (null 0.500)   n=7500/50000
ciciot2023: aggregate S_cov = 0.5123
   Benign         S_cov,c = 0.4919 (null 0.503)   n=6299/17999
   BruteForce     S_cov,c = 0.5098 (null 0.503)   n=1371/3919
   DDoS           S_cov,c = 0.4949 (null 0.500)   n=68476/195662
   DoS            S_cov,c = 0.4994 (null 0.499)   n=25215/72046
   Mirai          S_cov,c = 

In [5]:
# =============================================================================
# Cell 5 - THE TEST. Does class-conditional shift order class-conditional
# undercoverage where the aggregate cannot?
# =============================================================================
cov=[]
for env,f,rung in [('nslkdd','coverage_primary_nslkdd.csv',0.80),
                   ('ugr16','coverage_primary_ugr16.csv',None),
                   ('ciciot2023','coverage_primary_ciciot2023.csv',0.80)]:
    d=pd.read_csv(RD/f); d=d[(np.isclose(d.alpha,ALPHA))&(d.protocol=='SHC')]
    if 'feasible' in d.columns: d=d[d['feasible']]
    if rung is not None and 'rung' in d.columns: d=d[np.isclose(d['rung'],rung)]
    g=d.groupby('class',as_index=False)['coverage'].mean(); g['dataset']=env
    cov.append(g)
COV=pd.concat(cov,ignore_index=True); COV['undercoverage']=(1-ALPHA)-COV['coverage']
M=SC.merge(COV,on=['dataset','class'],how='inner')
M=M[M.estimable].copy()
print('CLASS-CONDITIONAL SHIFT vs CLASS-CONDITIONAL UNDERCOVERAGE')
print(M[['dataset','class','S_cov_class','S_cov_null','S_cov_aggregate','coverage','undercoverage']]
      .round(4).to_string(index=False))

print('\nORDERING POWER (Spearman against undercoverage):')
r_agg,p_agg=stats.spearmanr(M.S_cov_aggregate, M.undercoverage)
r_cls,p_cls=stats.spearmanr(M.S_cov_class,     M.undercoverage)
print(f'  aggregate S_cov          rho={r_agg:+.3f}  p={p_agg:.4f}')
print(f'  class-conditional S_cov,c rho={r_cls:+.3f}  p={p_cls:.4f}')

# excess over the permutation null is the honest version of "how much shift"
M['excess']=M.S_cov_class-M.S_cov_null
r_exc,p_exc=stats.spearmanr(M.excess.fillna(0), M.undercoverage)
print(f'  excess over null          rho={r_exc:+.3f}  p={p_exc:.4f}')

# and against the mechanism, for reference
ms=pd.read_csv(RD/'score_shift_explainability.csv'); ms['ds']=ms['dataset'].str.split(':').str[0]
msc=ms.groupby(['ds','class'],as_index=False)['score_KS'].mean().rename(columns={'ds':'dataset'})
M2=M.merge(msc,on=['dataset','class'],how='left')
if M2.score_KS.notna().sum()>3:
    r_ks,p_ks=stats.spearmanr(M2.dropna(subset=['score_KS']).score_KS,
                              M2.dropna(subset=['score_KS']).undercoverage)
    print(f'  score movement (reference) rho={r_ks:+.3f}  p={p_ks:.4f}')

print('\nVERDICT:')
if abs(r_cls) > abs(r_agg) + 0.15:
    print('  Class-conditional shift orders the outcome substantially better than the')
    print('  aggregate. The paper should replace the aggregate measure in every place it is')
    print('  used to explain a class-level outcome, and the sharper claim is available:')
    print('  aggregate shift statistics fail because the relevant shift is class-conditional.')
elif abs(r_cls) > abs(r_agg):
    print('  Class-conditional shift orders the outcome somewhat better. Report both and do')
    print('  not overstate the improvement.')
else:
    print('  Class-conditional shift does NOT order the outcome better than the aggregate.')
    print('  Feature distinguishability is then not the operative quantity at any resolution,')
    print('  and only score movement is. Report this against the reviewer\u2019s expectation.')


CLASS-CONDITIONAL SHIFT vs CLASS-CONDITIONAL UNDERCOVERAGE
   dataset       class  S_cov_class  S_cov_null  S_cov_aggregate  coverage  undercoverage
    nslkdd      Normal       0.7886      0.5011           0.8934    0.9268         0.0232
    nslkdd         DoS       0.9595      0.4989           0.8934    0.3988         0.5512
    nslkdd       Probe       0.9794      0.4966           0.8934    0.6278         0.3222
    nslkdd         R2L       0.9905      0.4921           0.8934    0.0298         0.9202
     ugr16  background       0.6696      0.4983           0.7129    0.9492         0.0009
     ugr16         dos       0.5424      0.5004           0.7129    0.9503        -0.0003
     ugr16 nerisbotnet       0.5782      0.4985           0.7129    0.9473         0.0027
     ugr16      scan11       0.9997      0.4982           0.7129    0.5350         0.4150
     ugr16      scan44       0.9861      0.4996           0.7129    0.7990         0.1510
ciciot2023      Benign       0.4919      

In [6]:
# =============================================================================
# Cell 6 - the UGR'16 case in isolation, since it is the paper's existing proof
# that an aggregate cannot work, then save and commit.
# =============================================================================
ug=M[M.dataset=='ugr16'].sort_values('undercoverage',ascending=False)
if len(ug):
    print("UGR'16: one aggregate value, five very different outcomes")
    print(ug[['class','S_cov_aggregate','S_cov_class','S_cov_null','coverage']].round(4).to_string(index=False))
    sp_=ug.S_cov_class.max()-ug.S_cov_class.min()
    print(f'\n  aggregate S_cov is a single number: {ug.S_cov_aggregate.iloc[0]:.4f}')
    print(f'  class-conditional values span {ug.S_cov_class.min():.4f} to {ug.S_cov_class.max():.4f}, a range of {sp_:.4f}')
    print('  the aggregate cannot distinguish classes that differ by that much')

SC.to_csv(RD/'class_conditional_scov.csv', index=False)
M.to_csv(RD/'class_conditional_scov_vs_coverage.csv', index=False)
(RD/'class_conditional_scov_verdict.json').write_text(json.dumps({
 'definition':'S_cov,c = cross-fitted AUROC of a domain classifier separating source X|Y=c '
              'from target X|Y=c, 5 folds, up to 20,000 rows per side',
 'null':'permutation reference from splitting the pooled per-class data at random',
 'min_rows_per_side':MIN_PER_SIDE,
 'spearman_vs_undercoverage':{'aggregate':float(r_agg),'class_conditional':float(r_cls),
                              'excess_over_null':float(r_exc)},
 'n_estimable':int(M.estimable.sum()),
 'cells':M.round(5).to_dict('records')}, indent=2, default=str))
print('\nsaved class_conditional_scov{,_vs_coverage}.csv and the verdict')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','step 6: class-conditional covariate shift; tests whether S_cov,c orders class-level undercoverage where the aggregate cannot')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


UGR'16: one aggregate value, five very different outcomes
      class  S_cov_aggregate  S_cov_class  S_cov_null  coverage
     scan11           0.7129       0.9997      0.4982    0.5350
     scan44           0.7129       0.9861      0.4996    0.7990
nerisbotnet           0.7129       0.5782      0.4985    0.9473
 background           0.7129       0.6696      0.4983    0.9492
        dos           0.7129       0.5424      0.5004    0.9503

  aggregate S_cov is a single number: 0.7129
  class-conditional values span 0.5424 to 0.9997, a range of 0.4573
  the aggregate cannot distinguish classes that differ by that much

saved class_conditional_scov{,_vs_coverage}.csv and the verdict
[main a2e82b4] step 6: class-conditional covariate shift; tests whether S_cov,c orders class-level undercoverage where the aggregate cannot
 5 files changed, 273 insertions(+), 734 deletions(-)
 rewrite notebooks/36_results_ledger.ipynb (99%)
 create mode 100644 notebooks/42_class_conditional_scov.ipynb
 creat